In [ ]:
dbutils.widgets.text("catalog", "sunny_bay_roastery")
catalog = dbutils.widgets.get("catalog")

# Optional prefix for object/folder names so several people can share one
# catalog and schema (empty by default — single-user runs are unchanged).
dbutils.widgets.text("prefix", "")
prefix = dbutils.widgets.get("prefix")

In [ ]:
import sys
from pathlib import Path

# The data-generation modules live alongside this notebook in the bundle
# (bundle/src/data). From bundle/src/notebooks, that resolves to the sibling
# "data" folder.
sys.path.append(str((Path.cwd().parent / "data").resolve()))

from generator import generate_all

In [ ]:
# Sunny Bay Roastery source systems (all written as raw files to the bronze Volume):
#
#   POS        pos_transactions, pos_transaction_lines   café receipts (CSV)
#   Webshop    web_orders (nested JSON), web_accounts    online shop
#   CRM        crm_customers, crm_loyalty_cards          customers and loyalty program
#   ERP        erp_products, erp_product_categories,     product and store master data
#              erp_product_prices, erp_stores
#   HR         hr_employees                              org chart
#   Reference  ref_public_holidays                       German public holidays
#
# Sales cover a rolling 24-month window that ends yesterday.

def main(catalog: str, prefix: str = ""):
    volume_path = f"/Volumes/{catalog}/bronze/raw"
    generate_all(spark, volume_path, prefix)

In [ ]:
def init_catalog(catalog: str = "sunny_bay_roastery"):
    # The catalog is created once in "Lab 0 - Intro" before this job runs.
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.bronze")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.silver")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.gold")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS `{catalog}`.bronze.raw")

In [ ]:
init_catalog(catalog)
main(catalog, prefix)

print(f"✅ Initialized the `{catalog}`!")